# Analista de Experiencia del Cliente: 29/06/2026

## Pregunta de negocio

**¿Cuál es la puntuación media dada por los usuarios a los alojamientos
turísticos y qué porcentaje de alojamientos tienen una evaluación
general superior a 80 en cada ciudad?**

Este notebook corresponde únicamente al perfil de
**Analista de Experiencia del Cliente** del Sprint 1.

## Desglose y preparación del análisis

### Variables principales

- `apartment_id`: identificador del alojamiento.
- `city`: ciudad del alojamiento.
- `review_scores_rating`: valoración general, en escala de 0 a 100.

### Métricas principales

De acuerdo con el documento de KPI del equipo, este análisis incluye:

- **KPI 3:** índice de satisfacción general.
- **KPI 4:** ítem con mayor satisfacción promedio.
- Puntuación media por ciudad.
- Número de alojamientos evaluados por ciudad.
- Porcentaje de alojamientos con puntuación superior a 80 por ciudad.

### Análisis complementario

Para el KPI 4 se comparan los siguientes ítems:

- `review_scores_accuracy`
- `review_scores_cleanliness`
- `review_scores_checkin`
- `review_scores_communication`
- `review_scores_location`

Los valores nulos de las puntuaciones no se imputan. Para cada KPI se
utilizan únicamente los alojamientos que disponen de una valoración
válida.

## Relación con el trabajo previo del equipo

Este notebook utiliza la tabla final obtenida después de
Data Understanding, EDA, Data Cleaning, Data Transformation y
Data Reduction.

En este análisis no se repiten tareas de limpieza ni transformación.
Solo se valida que la tabla final tenga la estructura necesaria y se
calculan los KPI de experiencia del cliente.

## 1. Importación de librerías

In [ ]:
from getpass import getpass

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

## 2. Conexión segura a MySQL

La contraseña se solicita durante la ejecución y no se guarda en el
notebook ni aparece en GitHub.

In [ ]:
db_user = input("Usuario de MySQL: ")
db_password = getpass("Contraseña de MySQL: ")
db_host = input("Host de MySQL: ")
db_port = input("Puerto [3306]: ") or "3306"
db_name = input("Nombre de la base de datos: ")
table_name = input("Nombre de la tabla final: ")

if not table_name.replace("_", "").isalnum():
    raise ValueError(
        "El nombre de la tabla contiene caracteres no válidos."
    )

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=db_user,
    password=db_password,
    host=db_host,
    port=int(db_port),
    database=db_name,
)

engine = create_engine(connection_url)

query = f"SELECT * FROM `{table_name}`"

df = pd.read_sql(query, engine)

engine.dispose()

print("Tabla utilizada:", table_name)
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

df.head()

## 3. Validación del dataset final

Esta validación no modifica los datos. Solo comprueba que la tabla
final contiene las columnas, escalas y registros necesarios para el
análisis.

In [ ]:
main_columns = [
    "apartment_id",
    "city",
    "review_scores_rating",
]

satisfaction_columns = [
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
]

missing_main_columns = [
    column
    for column in main_columns
    if column not in df.columns
]

missing_satisfaction_columns = [
    column
    for column in satisfaction_columns
    if column not in df.columns
]

print(
    "Columnas principales que faltan:",
    missing_main_columns,
)

print(
    "Columnas de satisfacción que faltan:",
    missing_satisfaction_columns,
)

In [ ]:
if missing_main_columns:
    raise ValueError(
        "Faltan columnas necesarias para responder "
        "a la pregunta de negocio."
    )

df[main_columns].info()

In [ ]:
duplicated_ids = (
    df["apartment_id"]
    .duplicated()
    .sum()
)

print(
    "Apartment ID duplicados en la tabla final:",
    duplicated_ids,
)

In [ ]:
df["review_scores_rating"].describe()

In [ ]:
invalid_rating = df[
    df["review_scores_rating"].notna()
    & ~df["review_scores_rating"].between(0, 100)
]

print(
    "Valores de rating fuera del rango 0-100:",
    len(invalid_rating),
)

In [ ]:
if not missing_satisfaction_columns:
    display(
        df[satisfaction_columns]
        .describe()
        .T
    )

## 4. Valores nulos relacionados con el análisis

Los nulos se revisan solo para conocer cuántos registros pueden
participar en cada KPI. No se rellenan ni se transforman en este
notebook.

In [ ]:
null_summary = (
    df[main_columns]
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

null_summary

## 5. Base analítica de experiencia del cliente

Para los KPI principales se utilizan solo los alojamientos con ciudad
y puntuación general válidas.

In [ ]:
df_customer = df.dropna(
    subset=[
        "city",
        "review_scores_rating",
    ]
).copy()

print(
    "Alojamientos con puntuación válida:",
    df_customer["apartment_id"].nunique(),
)

## 6. KPI 3 — Índice de satisfacción general

In [ ]:
general_satisfaction_index = round(
    df_customer["review_scores_rating"].mean(),
    2,
)

print(
    "Índice de satisfacción general:",
    general_satisfaction_index,
)

## 7. Puntuación media por ciudad

In [ ]:
average_rating_city = (
    df_customer
    .groupby("city")["review_scores_rating"]
    .mean()
    .round(2)
    .reset_index(
        name="average_rating"
    )
    .sort_values(
        "average_rating",
        ascending=False,
    )
)

average_rating_city

## 8. Porcentaje de alojamientos con puntuación superior a 80

El porcentaje se calcula solo sobre los alojamientos que tienen una
puntuación válida en cada ciudad.

In [ ]:
df_customer["rating_over_80"] = (
    df_customer["review_scores_rating"] > 80
)

percentage_over_80_city = (
    df_customer
    .groupby("city")["rating_over_80"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index(
        name="percentage_over_80"
    )
    .sort_values(
        "percentage_over_80",
        ascending=False,
    )
)

percentage_over_80_city

## 9. Número de alojamientos evaluados por ciudad

In [ ]:
rated_accommodations_city = (
    df_customer
    .groupby("city")["apartment_id"]
    .nunique()
    .reset_index(
        name="rated_accommodations"
    )
)

rated_accommodations_city

## 10. Cobertura de valoraciones

Este indicador complementario muestra qué porcentaje de los
alojamientos de cada ciudad tiene una valoración general disponible.

In [ ]:
total_accommodations_city = (
    df.groupby("city")["apartment_id"]
    .nunique()
    .reset_index(
        name="total_accommodations"
    )
)

rating_coverage_city = (
    total_accommodations_city
    .merge(
        rated_accommodations_city,
        on="city",
        how="left",
    )
)

rating_coverage_city["rating_coverage_pct"] = (
    rating_coverage_city["rated_accommodations"]
    / rating_coverage_city["total_accommodations"]
    * 100
).round(2)

rating_coverage_city

## 11. Tabla final de resultados

In [ ]:
customer_experience_summary = (
    average_rating_city
    .merge(
        percentage_over_80_city,
        on="city",
        how="left",
    )
    .merge(
        rating_coverage_city,
        on="city",
        how="left",
    )
    .sort_values(
        "average_rating",
        ascending=False,
    )
)

customer_experience_summary

## 12. KPI 4 — Ítem con mayor satisfacción promedio

Este bloque se ejecuta si las columnas necesarias para el KPI 4 están
disponibles en la tabla final.

In [ ]:
if not missing_satisfaction_columns:
    satisfaction_summary = (
        df[satisfaction_columns]
        .mean()
        .round(2)
        .sort_values(
            ascending=False
        )
        .rename(
            "average_score"
        )
        .reset_index(
            names="satisfaction_item"
        )
    )

    display(satisfaction_summary)

    best_satisfaction_item = satisfaction_summary.iloc[0]
    worst_satisfaction_item = satisfaction_summary.iloc[-1]

    print(
        "Ítem con mayor satisfacción promedio:",
        best_satisfaction_item["satisfaction_item"],
        "-",
        best_satisfaction_item["average_score"],
    )

    print(
        "Ítem con menor satisfacción promedio:",
        worst_satisfaction_item["satisfaction_item"],
        "-",
        worst_satisfaction_item["average_score"],
    )
else:
    print(
        "No se puede calcular el KPI 4 porque "
        "faltan columnas de satisfacción."
    )

## 13. Visualizaciones

In [ ]:
average_rating_city.plot(
    x="city",
    y="average_rating",
    kind="bar",
    figsize=(10, 5),
    legend=False,
)

plt.title(
    "Puntuación media de los alojamientos por ciudad"
)
plt.xlabel("Ciudad")
plt.ylabel("Puntuación media (0-100)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
percentage_over_80_city.plot(
    x="city",
    y="percentage_over_80",
    kind="bar",
    figsize=(10, 5),
    legend=False,
)

plt.title(
    "Alojamientos con puntuación superior a 80"
)
plt.xlabel("Ciudad")
plt.ylabel("Porcentaje (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
if not missing_satisfaction_columns:
    satisfaction_summary.sort_values(
        "average_score"
    ).plot(
        x="satisfaction_item",
        y="average_score",
        kind="barh",
        figsize=(9, 5),
        legend=False,
    )

    plt.title(
        "Puntuación media por ítem de satisfacción"
    )
    plt.xlabel("Puntuación media (0-10)")
    plt.ylabel("Ítem de satisfacción")
    plt.tight_layout()
    plt.show()

## 14. Resultados principales automáticos

In [ ]:
highest_rating_city = average_rating_city.iloc[0]
lowest_rating_city = average_rating_city.iloc[-1]

highest_percentage_city = percentage_over_80_city.iloc[0]
lowest_percentage_city = percentage_over_80_city.iloc[-1]

print(
    "Ciudad con mayor puntuación media:",
    highest_rating_city["city"],
    "-",
    highest_rating_city["average_rating"],
)

print(
    "Ciudad con menor puntuación media:",
    lowest_rating_city["city"],
    "-",
    lowest_rating_city["average_rating"],
)

print(
    "Mayor porcentaje superior a 80:",
    highest_percentage_city["city"],
    "-",
    highest_percentage_city["percentage_over_80"],
)

print(
    "Menor porcentaje superior a 80:",
    lowest_percentage_city["city"],
    "-",
    lowest_percentage_city["percentage_over_80"],
)

## 15. Conclusiones

Esta sección se completará después de ejecutar el notebook con la tabla
final del equipo.

La conclusión debe explicar:

- El índice de satisfacción general.
- Las diferencias entre ciudades.
- El porcentaje de alojamientos con puntuación superior a 80.
- La cobertura de valoraciones.
- El ítem mejor y peor valorado.
- Las posibles implicaciones para mejorar la experiencia del cliente.

## 16. Exportación de resultados

La tabla final se puede guardar para utilizarla en el dashboard o en la
presentación del Sprint.

In [ ]:
output_path = (
    "../Data/customer_experience_summary.csv"
)

# Descomentar cuando se quiera guardar el resultado.
# customer_experience_summary.to_csv(
#     output_path,
#     index=False,
# )

## Opción de respaldo sin conexión

Si no hay acceso al servidor, se puede utilizar una copia CSV guardada
en la carpeta `Data` del repositorio.

In [ ]:
# Opción de respaldo:
# df = pd.read_csv(
#     "../Data/tourist_accommodation_reduced.csv"
# )